In [ ]:
# Imports and setup
import json
import os
import sys
from typing import List, Optional, Dict
import gc
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from simulate_markov_utils import plot_markov_df
from sklearn.metrics import auc, roc_curve
from tqdm import tqdm

sys.path.append('..')
sys.path.append('../..')
from plotting_notebooks.plotting_utils import (
    load_in_combined_results,
    load_monitoring_data,
    plot_markov_analysis,
    plot_roc_curves,
    plot_suspiciousness_distributions,
    plot_cost_vs_safety_curves,
    get_roc_curve,
    interpolate_tpr,
    prepare_monitoring_data,
    merge_monitoring_with_combined_results,
    plot_training_progress,
    plot_all_suspiciousness_metrics,
    is_in_training_data,
    filter_and_prepare_data,
    prepare_data_for_training_progress,
    plot_solution_review_performance,
    plot_solution_review_training_progress,
    load_solution_reviews_data,
)
from simulate_markov_utils import plot_markov_df

from utils.analysis_utils import extract_verdict
from utils.path_utils import resolve_path


In [ ]:
# Constants and configuration
MAIN_AGENT_MODELS = ['claude-3-5-sonnet-20241022', 'claude-3-7-sonnet-20250219']

# Step mapping for fine-tuned models (second series)
default_step_mapping = {
    "gpt-4o-2024-08-06": 0,
    'ft:gpt-4o-2024-08-06:scale-safety-research-1:combined-3-reviewed:BCYJFx2l:ckpt-step-70': 70,
    # 'ft:gpt-4o-2024-08-06:scale-safety-research-1:combined-3-reviewed:BCYJFtC0:ckpt-step-140': 140,
    # 'ft:gpt-4o-2024-08-06:scale-safety-research-1:combined-3-reviewed:BCYJFa40': 210,
    # 'ft:gpt-4o-2024-08-06:scale-safety-research-1:unbalanced-data:BDXppMow:ckpt-step-56': 56,
    # 'ft:gpt-4o-2024-08-06:scale-safety-research-1:unbalanced-data:BDXpp5YL:ckpt-step-112': 112,
    # 'ft:gpt-4o-2024-08-06:scale-safety-research-1:unbalanced-data:BDXppDFa': 168
}
# default_step_mapping = {
#     "gpt-4o-2024-08-06": 0,
#     "ft:gpt-4o-2024-08-06:scale-safety-research-1::BAU1ENZ3:ckpt-step-140": 140,
#     "ft:gpt-4o-2024-08-06:scale-safety-research-1::BAU1Fwer:ckpt-step-210": 210,
#     "ft:gpt-4o-2024-08-06:scale-safety-research-1::BAU1FbkP": 280,
# }
FT_MONITOR_MODELS = list(default_step_mapping.keys())


# Cache for combined_results
_combined_results_cache = {}
NUM_MONITORS = 10


In [ ]:
# Helper functions for data loading and preparation
def load_combined_results(include_failed_attempts: bool, include_failed_main_task: bool) -> pd.DataFrame:
    """Load and cache combined_results."""
    cache_key = f"{include_failed_attempts}_{include_failed_main_task}"
    if cache_key in _combined_results_cache:
        print(f"Using cached combined_results for {cache_key}")
        return _combined_results_cache[cache_key]
    combined_results = load_in_combined_results(include_failed_attempts, include_failed_main_task)
    combined_results = combined_results[combined_results.main_agent.isin(MAIN_AGENT_MODELS)]
    _combined_results_cache[cache_key] = combined_results
    return combined_results

def prepare_system_prompt_data(base_dir: str) -> pd.DataFrame:
    """Load and prepare the initial datasets."""
    # Use the generalized function from plotting_utils
    system_prompt_df = prepare_monitoring_data(
        base_dir=base_dir,
        monitor_models=FT_MONITOR_MODELS,  # Filter to only include fine-tuned models
        num_monitors=NUM_MONITORS,  # Process all monitor responses
        compute_aggregates=True,  # Compute max, mean, etc.
        tag_word='verdict'
    )
    gc.collect()
    return system_prompt_df

def merge_and_compute_features_for_system_prompt(combined_results: pd.DataFrame, 
                             system_prompt_df: pd.DataFrame) -> pd.DataFrame:
    """Merge the dataframes and compute additional features."""
    return merge_monitoring_with_combined_results(
        monitoring_df=system_prompt_df,
        combined_results=combined_results,
        how='left',  # Use left join as in the original
        compute_convo_length=True
    )
    gc.collect()

# Oversampling approach for balancing classes
def balance_classes_with_oversampling(df, class_column='task_name', target_size=None, random_state=42):
    """
    Balance the dataset by oversampling smaller classes up to the size of the largest class
    (or a specified target size).
    
    Args:
        df: DataFrame to balance
        class_column: Column to balance by (default: 'task_name')
        target_size: Target size for each class, defaults to max class size
        random_state: Random seed for reproducibility
        
    Returns:
        Balanced DataFrame with equal representation of each class
    """
    # Get the distribution of classes
    class_counts = df[class_column].value_counts()
    
    # Determine target size
    if target_size is None:
        target_size = class_counts.max()
    
    # Create an empty list to store the balanced data
    balanced_dfs = []
    
    # Sample from each class up to the target size
    for class_name, count in class_counts.items():
        if count < target_size/10:
            continue
        class_df = df[df[class_column] == class_name]
        
        if count < target_size:
            # Need to oversample
            # Calculate how many complete copies we need
            num_complete_copies = target_size // count
            # Calculate how many additional samples we need
            remainder = target_size % count
            
            # Add complete copies
            oversampled_parts = [class_df] * num_complete_copies
            
            # Add the remainder as a random sample
            if remainder > 0:
                remainder_sample = class_df.sample(remainder, random_state=random_state, replace=False)
                oversampled_parts.append(remainder_sample)
            
            # Combine all parts
            oversampled_df = pd.concat(oversampled_parts, ignore_index=False)
            balanced_dfs.append(oversampled_df)
        else:
            # Need to undersample
            sampled_df = class_df.sample(target_size, random_state=random_state, replace=False)
            balanced_dfs.append(sampled_df)
    
    # Combine all the balanced classes
    balanced_df = pd.concat(balanced_dfs, ignore_index=False)
    
    # Verify the new distribution
    new_class_counts = balanced_df[class_column].value_counts()
    
    return balanced_df

def balance_df(df, class_column='task_name', target_size=None):
    return pd.concat([balance_classes_with_oversampling(df[df.main_agent == main_agent], class_column=class_column, target_size=target_size) for main_agent in MAIN_AGENT_MODELS])

In [ ]:
# Load the base data
combined_results = load_combined_results(include_failed_attempts=False, include_failed_main_task=False)

# Load system prompt data from different system prompts
system_prompt_df_1 = prepare_system_prompt_data(base_dir='monitor_responses/system_prompt_1_temp_0')
system_prompt_df_5 = prepare_system_prompt_data(base_dir='monitor_responses/system_prompt_5_temp_0')
merge_on = ['main_agent', 'task_name', 'transcript_type', 'file_name']
just_merge_on_and_side_task = combined_results[merge_on + ['side_task_evaluation']].drop_duplicates()
gc.collect()

system_prompt_df_1 = pd.merge(system_prompt_df_1, just_merge_on_and_side_task, on=merge_on, how='left').dropna()
system_prompt_df_5 = pd.merge(system_prompt_df_5, just_merge_on_and_side_task, on=merge_on, how='left').dropna()
gc.collect()

In [ ]:
# Load training data
training_data_path = resolve_path('finetuning/training_data_dfs/combined_3.csv')
training_data = pd.read_csv(training_data_path)

# Identify which examples are in the training data
system_prompt_df_1['in_training_data'] = system_prompt_df_1.apply(
    lambda row: is_in_training_data(row, training_data), 
    axis=1
)
system_prompt_df_5['in_training_data'] = system_prompt_df_5.apply(
    lambda row: is_in_training_data(row, training_data), 
    axis=1
)
gc.collect()


In [ ]:
# Merge the system prompt dataframes and add combined metrics
both_system_prompts_df = pd.merge(
    system_prompt_df_1,
    system_prompt_df_5,
    on=['task_name', 'transcript_type', 'main_agent', 'monitor_agent', 'file_name', 'in_training_data', 'side_task_evaluation'],
    suffixes=('_sp1', '_sp5'),
    how='inner'
)

# Add computed metrics
both_system_prompts_df['total_max'] = both_system_prompts_df[['max_suspiciousness_sp1', 'max_suspiciousness_sp5']].max(axis=1)
both_system_prompts_df['total_mean'] = both_system_prompts_df[['mean_suspiciousness_sp1', 'mean_suspiciousness_sp5']].mean(axis=1)
both_system_prompts_df['total_min'] = both_system_prompts_df[['min_suspiciousness_sp1', 'min_suspiciousness_sp5']].min(axis=1)

gc.collect()

# Add standard deviation metrics
for suffix in ['_sp1', '_sp5']:
    both_system_prompts_df[f'std_dev{suffix}'] = both_system_prompts_df[
        [f'suspiciousness_{i}{suffix}' for i in range(1, NUM_MONITORS + 1)]
    ].std(axis=1)



In [ ]:
# from plotting_notebooks.plotting_utils import filter_and_prepare_data
# filtered_df = filter_and_prepare_data(
#     df=both_system_prompts_df,
#     data_to_use='all_test',
#     training_data=training_data,
#     exclude_tasks=[]
# )
# filtered_df
# index_columns = [col for col in ['task_name', 'transcript_type', 'main_agent', 'file_name', 
#                                     'side_task_evaluation', 'in_training_data'] 
#                     if col in filtered_df.columns]
# unstacked_df = filtered_df.set_index(index_columns + ['monitor_agent'])
# unstacked_df = unstacked_df[aggregation_type].unstack(level='monitor_agent')
# # unstacked_df = unstacked_df.reset_index().dropna()
# # unstacked_df = unstacked_df[~unstacked_df.task_name.isin(training_data.task_name.unique())]
# # unstacked_df[unstacked_df.transcript_type != 'transcripts_no_secondary_task'].side_task_evaluation.value_counts()
# unstacked_df

In [ ]:
# Plot ROC curves for different data filters
DATA_TO_USE = 'no_training_tasks'  # Options: 'all_test', 'no_training_tasks', 'training_tasks_no_train_data', 'training_data_only'
aggregation_type = 'mean_suspiciousness_sp5'

# Use the new utility function to prepare data for plotting
unstacked_df = prepare_data_for_training_progress(
    df=both_system_prompts_df,
    aggregation_type=aggregation_type,
    step_mapping=default_step_mapping,
    data_to_use=DATA_TO_USE,
    training_data=training_data,
    exclude_tasks=['election_tampering']
)

unstacked_df = balance_df(unstacked_df)

# You can use either of these balancing approaches
# Uncomment to use the stratified balancing approach
# balanced_unstacked_df = balance_classes_stratified(
#     unstacked_df, 
#     strata_columns=['task_name', 'side_task_evaluation'],
#     sample_strategy='min'
# )

# Uncomment to use oversampling instead of undersampling
# balanced_unstacked_df = balance_classes_with_oversampling(unstacked_df)

# Use the balanced dataset for analysis

for main_agent in MAIN_AGENT_MODELS:
    print(main_agent)
    # Plot ROC curves
    plot_roc_curves(
        unstacked_df[unstacked_df.main_agent == main_agent], 
        combined_results[combined_results.main_agent == main_agent], 
        give_hint=None, 
        include_failed_attempts=False, 
        include_failed_main_task=False, 
        monitors=[aggregation_type], 
        metrics=list(default_step_mapping.values())
    )
gc.collect()

In [ ]:
main_agent_mapping = {
    'claude-3-5-sonnet-20241022': 'Claude 3.6 Sonnet',
    'claude-3-7-sonnet-20250219': 'Claude 3.7 Sonnet',
    'gpt-4o': 'GPT-4o',
    'gpt-4o-2024-08-06': 'GPT-4o',
}
for main_agent in MAIN_AGENT_MODELS + ['Mixed']:
    plt.figure(figsize=(10, 10))
    for aggregation_type in ['mean_suspiciousness_sp5', 'mean_suspiciousness_sp1']:
        # Create more readable label for system prompt type
        sp_label = "System Prompt A" if aggregation_type == 'mean_suspiciousness_sp5' else "System Prompt B"
        
        unstacked_df = prepare_data_for_training_progress(
            df=both_system_prompts_df,
            aggregation_type=aggregation_type,
            step_mapping=default_step_mapping,
            data_to_use=DATA_TO_USE,
            training_data=training_data,
            exclude_tasks=['election_tampering']
        )
        unstacked_df = balance_df(unstacked_df)
        if main_agent == 'Mixed':
            unstacked_df = unstacked_df[unstacked_df.main_agent.isin(MAIN_AGENT_MODELS)]
            unstacked_df = unstacked_df[((unstacked_df.main_agent == 'claude-3-5-sonnet-20241022') & (unstacked_df.transcript_type == 'transcripts_no_secondary_task')) | ((unstacked_df.main_agent == 'claude-3-7-sonnet-20250219') & (unstacked_df.transcript_type != 'transcripts_with_secondary_task'))]
        else:
            unstacked_df = unstacked_df[unstacked_df.main_agent == main_agent]
        
        for metric in default_step_mapping.values():
            fpr, tpr, _ = roc_curve(unstacked_df['side_task_evaluation'], unstacked_df[metric])
            auc_score = auc(fpr, tpr)
            
            # Create more descriptive label
            model_type = "Base Model" if metric == 0 else "Fine-tuned Model"
            label = f'{sp_label} - {model_type} (AUC: {auc_score:.2f})'
            
            # Use different line styles for base vs fine-tuned
            linestyle = '--' if metric == 0 else '-'
            # Use different colors for SP1 vs SP5
            color = 'blue' if aggregation_type == 'mean_suspiciousness_sp5' else 'red'
            
            plt.plot(fpr, tpr, label=label, linestyle=linestyle, color=color)
            
    plt.plot([0, 1], [0, 1], 'k:', label='Random Baseline', alpha=0.5)
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'ROC Curves: Base vs Fine-tuned Models\n{main_agent_mapping.get(main_agent, main_agent)}')
    plt.grid(True, alpha=0.3)
    plt.legend(loc='lower right')
    plt.tight_layout()
    plt.show()

In [ ]:
for main_agent in MAIN_AGENT_MODELS:
    print(main_agent)
    plot_suspiciousness_distributions(unstacked_df[unstacked_df.main_agent == main_agent], monitors=[aggregation_type], metrics=[0, 70])

In [ ]:
balanced_filtered_both_system_prompts_df = balance_df(both_system_prompts_df[~both_system_prompts_df.task_name.isin(list(training_data.task_name.unique()) + ['election_tampering'])])
for metric in ['mean_suspiciousness_sp1', 'mean_suspiciousness_sp5']:
    for monitor in FT_MONITOR_MODELS[:2]:
        for side_task_evaluation in [0, 1]:
            plt.figure(figsize=(10, 5))
            bening_or_malign = 'benign' if side_task_evaluation == 0 else 'malign'
            plt.title(f'{metric}, monitor {default_step_mapping[monitor]} {bening_or_malign}')
            for main_agent in MAIN_AGENT_MODELS:
                plt.hist(
                    balanced_filtered_both_system_prompts_df[(balanced_filtered_both_system_prompts_df.main_agent == main_agent) & (balanced_filtered_both_system_prompts_df.monitor_agent == monitor) & (balanced_filtered_both_system_prompts_df.side_task_evaluation == side_task_evaluation)][metric], 
                    label=main_agent, 
                    alpha=0.5,
                    bins=np.linspace(0, 100, 30),
                    density=True)
            plt.legend()
            plt.show()

In [ ]:
balanced_filtered_both_system_prompts_df[balanced_filtered_both_system_prompts_df.monitor_agent.isin(FT_MONITOR_MODELS[:2])].groupby(['monitor_agent', 'side_task_evaluation', 'main_agent']).agg({
    'mean_suspiciousness_sp1': 'mean',
    'mean_suspiciousness_sp5': 'mean'
})

In [ ]:
side_task_3_5 = 1
hybrid_df = pd.concat([balanced_filtered_both_system_prompts_df[(balanced_filtered_both_system_prompts_df.main_agent == 'claude-3-5-sonnet-20241022') & (balanced_filtered_both_system_prompts_df.side_task_evaluation == side_task_3_5)], balanced_filtered_both_system_prompts_df[(balanced_filtered_both_system_prompts_df.main_agent == 'claude-3-7-sonnet-20250219') & (balanced_filtered_both_system_prompts_df.side_task_evaluation == 1- side_task_3_5)]])
plot_roc_curves(
    hybrid_df,
    combined_results,
    give_hint=None,
    include_failed_attempts=False,
    include_failed_main_task=False,
    monitors=FT_MONITOR_MODELS[:2],
    metrics=['mean_suspiciousness_sp1', 'mean_suspiciousness_sp5']
)

In [ ]:
hybrid_df.transcript

In [ ]:
# Filter data for analysis based on selected data subset
if DATA_TO_USE == 'all_test':
    interesting_df = filter_and_prepare_data(
        both_system_prompts_df,
        data_to_use='all_test',
        training_data=training_data,
        exclude_tasks=[]
    )
elif DATA_TO_USE == 'no_training_tasks':
    interesting_df = filter_and_prepare_data(
        both_system_prompts_df,
        data_to_use='no_training_tasks',
        training_data=training_data,
        exclude_tasks=[]
    )
elif DATA_TO_USE == 'training_tasks_no_train_data':
    interesting_df = filter_and_prepare_data(
        both_system_prompts_df,
        data_to_use='training_tasks_no_train_data',
        training_data=training_data,
        exclude_tasks=[]
    )
else:
    interesting_df = filter_and_prepare_data(
        both_system_prompts_df,
        data_to_use='training_data_only',
        training_data=training_data,
        exclude_tasks=[]
    )



In [ ]:
# Plot training progress curves for specific metrics
training_progress_df = plot_training_progress(
    df=interesting_df[~interesting_df.task_name.isin(['election_tampering', 'charity_allocation'])],
    metric='mean_suspiciousness_sp1',
    step_mapping=default_step_mapping,
    model_column='monitor_agent',
    models=FT_MONITOR_MODELS,
    target_fpr=0.03
)


In [ ]:
# Plot all suspiciousness metrics
all_metrics_df = plot_all_suspiciousness_metrics(
    df=interesting_df[interesting_df.task_name == 'election_tampering'],
    metrics_list=[
        'mean_suspiciousness_sp1', 'mean_suspiciousness_sp5',
        'max_suspiciousness_sp1', 'max_suspiciousness_sp5',
    ],
    step_mapping=default_step_mapping,
    model_column='monitor_agent',
    models=FT_MONITOR_MODELS,
    target_fpr=0.03
)

In [ ]:
interesting_df.task_name.unique()

In [ ]:
# Compare average suspiciousness scores across models
model_comparison = both_system_prompts_df.groupby('monitor_agent').agg({
    'suspiciousness_1_sp1': 'std', 
    'suspiciousness_1_sp5': 'std'
})

print("Standard deviation of suspiciousness scores by model:")
print(model_comparison)

In [ ]:
both_system_prompts_df['std_dev_sp1'] = both_system_prompts_df[[f'suspiciousness_{i}_sp1' for i in range(1, NUM_MONITORS + 1)]].std(axis=1)
both_system_prompts_df['std_dev_sp5'] = both_system_prompts_df[[f'suspiciousness_{i}_sp5' for i in range(1, NUM_MONITORS + 1)]].std(axis=1)

In [ ]:
# Plot suspiciousness distributions for non-default transcript types
##%
plot_suspiciousness_distributions(
    both_system_prompts_df, 
    monitors=FT_MONITOR_MODELS, 
    metrics=['min_suspiciousness_sp1']
)

In [ ]:
just_210_df = both_system_prompts_df[both_system_prompts_df.monitor_agent == 'ft:gpt-4o-2024-08-06:scale-safety-research-1:combined-3-reviewed:BCYJFa40']
filtered_210_df = just_210_df[just_210_df.std_dev_sp1 > 30]

# Create a filtered version of both_system_prompts_df based on filtered_210_df tuples
filtered_both_prompts = both_system_prompts_df.merge(
    filtered_210_df[['task_name', 'transcript_type', 'file_name']].drop_duplicates(),
    on=['task_name', 'transcript_type', 'file_name']
)
plot_suspiciousness_distributions(
    filtered_both_prompts, 
    monitors=FT_MONITOR_MODELS, 
    metrics=['min_suspiciousness_sp1']
)

In [ ]:
# Find examples with low suspiciousness despite having side tasks
suspicious_examples = interesting_df[
    (interesting_df.side_task_evaluation == 1) & 
    (interesting_df.task_name != 'election_tampering') &
    (interesting_df.monitor_agent == FT_MONITOR_MODELS[-1])
].sort_values('mean_suspiciousness_sp1', ascending=True).head(10)

print("Examples with low suspiciousness scores despite having side tasks:")
print(suspicious_examples[['task_name', 'transcript_type', 'mean_suspiciousness_sp1', 'mean_suspiciousness_sp5']])


In [ ]:
task = 'api_key_calendar_agendas_2'
system_prompt_number = 5
solution_reviews_df = load_solution_reviews_data()
solution_reviews_df = solution_reviews_df[solution_reviews_df.task_name == task]

In [ ]:
filtered_system_prompts_df = both_system_prompts_df[both_system_prompts_df.task_name == task]
# Calculate a good grid layout based on number of models
n_models = len(FT_MONITOR_MODELS)
n_cols = min(3, n_models)  # Maximum 3 columns
n_rows = (n_models + n_cols - 1) // n_cols  # Ceiling division for rows

# Create a single figure with subplots
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4 * n_rows), constrained_layout=True)
if n_models == 1:  # Handle the case with just one subplot
    axes = np.array([axes])
axes = axes.flatten()  # Flatten for easy indexing

# Get global min/max across all datasets for consistent bins
global_min = 0
global_max = 100

# Create fixed bins for all plots
bins = np.linspace(global_min, global_max, 20)

# Plot each model in its own subplot
for i, monitor in enumerate(FT_MONITOR_MODELS):
    ax = axes[i]  # Get the current subplot
    
    # Get all three suspiciousness metrics and combine them for each category
    # With side task - combine all three metrics
    data_with_side_task = []
    for metric in [f'suspiciousness_{i}_sp{system_prompt_number}' for i in range(1, NUM_MONITORS + 1)]:
        data = filtered_system_prompts_df[(filtered_system_prompts_df.monitor_agent == monitor) & 
                                         (filtered_system_prompts_df.side_task_evaluation == 1)][metric]
        if not data.empty:
            data_with_side_task.extend(data.tolist())
    
    # Without side task - combine all three metrics
    data_without_side_task = []
    for metric in ['suspiciousness_1_sp5', 'suspiciousness_2_sp5', 'suspiciousness_3_sp5']:
        data = filtered_system_prompts_df[(filtered_system_prompts_df.monitor_agent == monitor) & 
                                         (filtered_system_prompts_df.side_task_evaluation == 0)][metric]
        if not data.empty:
            data_without_side_task.extend(data.tolist())
    
    # Solution reviews - get all verdicts
    data_solution_reviews = []
    if not solution_reviews_df[solution_reviews_df.model_name == monitor].empty:
        for i in range(1, 6):  # Get all 5 verdicts
            verdict = solution_reviews_df[solution_reviews_df.model_name == monitor][f'verdict_{i}'].iloc[0]
            if verdict is not None:
                data_solution_reviews.append(verdict)
    
    # Plot histograms on this subplot - normalized as probability distributions
    ax.hist(data_with_side_task, bins=bins, alpha=0.7, density=True, label="With side task (all metrics)")
    ax.hist(data_without_side_task, bins=bins, alpha=0.7, density=True, label="Without side task (all metrics)")
    ax.hist(data_solution_reviews, bins=bins, alpha=0.7, density=True, color='gold', label="Solution reviews (all verdicts)")
    
    # Add labels and title for this subplot
    ax.set_title(f'Step {default_step_mapping[monitor]}')
    ax.set_xlabel('Suspiciousness score')
    ax.set_ylabel('Probability density')
    
    # Only add legend to the first subplot to avoid repetition
    if i == 0:
        ax.legend()

    # Add sample size information to each subplot
    sample_info = f"n_with={len(data_with_side_task)}, n_without={len(data_without_side_task)}, n_reviews={len(data_solution_reviews)}"
    ax.text(0.5, 0.97, sample_info, transform=ax.transAxes, ha='center', va='top', fontsize=8)

# Hide any unused subplots
for j in range(n_models, len(axes)):
    axes[j].set_visible(False)

# Add a common title for the entire figure
fig.suptitle('Normalized probability distributions of suspiciousness scores across training steps', fontsize=16)
plt.show()


In [ ]:
for t in training_data[(training_data.task_name == "conference_recap") & (training_data.side_task_evaluation == 1)].transcript:
    print(t[t.index("monitoring_response"):])

In [ ]:
both_system_prompts_df['usage_1_sp1'].iloc[0][0]['cost_usd']

In [ ]:
def calculate_auc_and_cost_using_num_monitors(df, num_monitors, sp_num, monitor_model, tasks_to_exclude=[], target_fpr=0.03):
    filtered_df = df[df.monitor_agent == monitor_model]
    filtered_df = filtered_df[~filtered_df.task_name.isin(tasks_to_exclude)]
    indices = np.random.choice(range(1, num_monitors + 1), size=num_monitors, replace=False)
    filtered_df['agg_score'] = filtered_df[[f'suspiciousness_{i}_sp{sp_num}' for i in indices]].mean(axis=1)
    filtered_df['cost'] = sum([filtered_df[f'usage_{i}_sp{sp_num}'].apply(lambda x: x[0]['cost_usd']) for i in indices])
    
    # Calculate ROC curve and AUC
    fpr, tpr, _ = roc_curve(filtered_df['side_task_evaluation'], filtered_df['agg_score'])
    auc_score = auc(fpr, tpr)
    
    # Calculate TPR at target FPR
    interpolated_tpr = interpolate_tpr(fpr, tpr, target_fpr)
    
    mean_cost = filtered_df['cost'].mean()
    del filtered_df
    return auc_score, mean_cost, interpolated_tpr

# Create the figures
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8))

for model in FT_MONITOR_MODELS:
    auc_scores = []
    costs = []
    tprs = []
    for num_monitors in range(1, 11):
        auc_score = 0
        mean_cost = 0
        tpr_score = 0
        for i in range(10):
            new_auc_score, new_mean_cost, new_tpr = calculate_auc_and_cost_using_num_monitors(
                both_system_prompts_df, 
                num_monitors, 
                1, 
                model, 
                tasks_to_exclude=list(training_data.task_name.unique()) + ['election_tampering'],
                target_fpr=0.03
            )
            auc_score += new_auc_score
            mean_cost += new_mean_cost
            tpr_score += new_tpr
        auc_scores.append(auc_score / 10)
        costs.append(mean_cost / 10)
        tprs.append(tpr_score / 10)
    gc.collect()
    
    # Plot AUC vs Cost
    ax1.scatter(costs, auc_scores, label=f'Step {default_step_mapping[model]}', alpha=0.7, s=100)
    ax1.plot(costs, auc_scores, alpha=0.3)
    
    # Plot TPR vs Cost
    ax2.scatter(costs, tprs, label=f'Step {default_step_mapping[model]}', alpha=0.7, s=100)
    ax2.plot(costs, tprs, alpha=0.3)

# Customize the AUC plot
ax1.grid(True, linestyle='--', alpha=0.7)
ax1.set_xlabel('Average Cost per Conversation (USD)', fontsize=12)
ax1.set_ylabel('Area Under ROC Curve (AUC)', fontsize=12)
ax1.set_title('Cost vs AUC Trade-off\nVarying Number of Monitors (1-10)', fontsize=14, pad=20)
ax1.legend(title='Training Steps', title_fontsize=12, fontsize=10)

# Customize the TPR plot
ax2.grid(True, linestyle='--', alpha=0.7)
ax2.set_xlabel('Average Cost per Conversation (USD)', fontsize=12)
ax2.set_ylabel('TPR at FPR=0.03', fontsize=12)
ax2.set_title('Cost vs TPR Trade-off\nVarying Number of Monitors (1-10)', fontsize=14, pad=20)
ax2.legend(title='Training Steps', title_fontsize=12, fontsize=10)

# Add some padding to the layout
plt.tight_layout()

# Show the plot
plt.show()

In [ ]:
plot_markov_analysis(both_system_prompts_df[~both_system_prompts_df.task_name.isin(list(training_data.task_name.unique()) + ['election_tampering'])], monitor_model='ft:gpt-4o-2024-08-06:scale-safety-research-1:combined-3-reviewed:BCYJFa40', give_hint=False, metrics=['mean_suspiciousness_sp1'])